In [1]:
import torch
import torch.nn as nn
import numpy as np

with open("full_speed_ahead.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Characters:", len(text))

chars = sorted(list(set(text)))
vocab_size = len(chars)

char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}

print("Vocabulary size:", vocab_size)

seq_length = 50

X = []
Y = []

for i in range(len(text) - seq_length):
    X.append([char_to_idx[c] for c in text[i:i+seq_length]])
    Y.append(char_to_idx[text[i+seq_length]])

X = torch.tensor(X, dtype=torch.long)
Y = torch.tensor(Y, dtype=torch.long)

print("Training samples:", len(X))

Characters: 2285
Vocabulary size: 42
Training samples: 2235


In [2]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, 64)

        self.lstm = nn.LSTM(
            input_size=64,
            hidden_size=128,
            num_layers=2,
            batch_first=True
        )

        self.fc = nn.Linear(128, vocab_size)

    def forward(self, x):

        x = self.embedding(x)

        out, _ = self.lstm(x)

        out = out[:, -1, :]

        return self.fc(out)


model = CharLSTM(vocab_size)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(model)

CharLSTM(
  (embedding): Embedding(42, 64)
  (lstm): LSTM(64, 128, num_layers=2, batch_first=True)
  (fc): Linear(in_features=128, out_features=42, bias=True)
)


In [5]:
epochs = 100

for epoch in range(epochs):

    optimizer.zero_grad()

    outputs = model(X)

    loss = criterion(outputs, Y)

    loss.backward()

    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Loss = {loss.item():.4f}"
        )

Epoch [10/100] Loss = 2.5186
Epoch [20/100] Loss = 2.3138
Epoch [30/100] Loss = 2.1101
Epoch [40/100] Loss = 1.9219
Epoch [50/100] Loss = 1.7505
Epoch [60/100] Loss = 1.5945
Epoch [70/100] Loss = 1.4490
Epoch [80/100] Loss = 1.3154
Epoch [90/100] Loss = 1.1962
Epoch [100/100] Loss = 1.1024


In [6]:
def generate_text(seed,
                  length=500,
                  temperature=0.8):

    model.eval()

    generated = seed

    for _ in range(length):

        seq = generated[-seq_length:]

        if len(seq) < seq_length:
            seq = " " * (seq_length - len(seq)) + seq

        x = torch.tensor(
            [[char_to_idx.get(c, 0) for c in seq]],
            dtype=torch.long
        )

        with torch.no_grad():
            logits = model(x)[0]

        probs = torch.softmax(
            logits / temperature,
            dim=0
        ).numpy()

        next_idx = np.random.choice(
            vocab_size,
            p=probs
        )

        generated += idx_to_char[next_idx]

    return generated


seed = "Full speed ahead"

print(
    generate_text(
        seed=seed,
        length=500,
        temperature=0.7
    )
)

Full speed ahead tht aun the seon the ryou fter mon rinhe I's I't'n bor doung lou seeid I fer yoig ting You than aut lou see sead et tule ey 'rees yuse feat teree now wou set let the lilling foid ming I'm I'm mnm ingt thint al'ded yoigh taung I'd domd You ghith I'd kaus fpead you lound mead I've roight you lise feat set for dead oun fead, he you the ellu Is me coing me coinn for  ead thing bear you cingt yow oul seed Se here rode wowsy gow gou sete he rade the I sted now thee ber seyin You though I lol tust rea
